In [38]:
%load_ext autoreload
%autoreload 2
import torch

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel

device = C.get_device()
print(f"Using device: {device}")

checkpoint_path = "../trained_models/ctc_specaugment_70epochs.pt"

checkpoint = torch.load(checkpoint_path, map_location=device)

model = CTCModel().to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

Using device: cpu


CTCModel(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (7): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (14):

In [54]:

from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, WLIST
WLIST1000 = [
    # Zwierzęta
    'pies', 'ptak', 'rybka', 'chomik', 'krowa', 'koń', 'świnia', 'owca', 'koza', 'kura',
    'kaczka', 'gęś', 'indyk', 'wąż', 'jaszczurka', 'żaba', 'pająk', 'mucha', 'komar', 'osa',
    'pszczoła', 'mrówka', 'motyl', 'tygrys', 'lew', 'słoń', 'żyrafa', 'małpa', 'niedźwiedź', 'wilk',
    'lis', 'zając', 'królik', 'jeleń', 'sarna', 'dzik', 'wiewiórka', 'jeż', 'nietoperz', 'rekin',
    'wieloryb', 'delfin', 'foka', 'pingwin', 'orzeł', 'sokół', 'gołąb', 'wróbel', 'sikorka', 'kruk',
    'sowa', 'bocian', 'łabędź', 'karp', 'szczupak', 'śledź', 'łoś', 'żubr', 'ryś', 'borsuk',
    'kret', 'bóbr', 'wydra', 'kuna', 'łasica', 'szop', 'hipopotam', 'nosorożec', 'zebra', 'krokodyl',
    'aligator', 'żółw', 'skorpion', 'stonoga', 'dżdżownica', 'ślimak', 'rak', 'krab', 'ośmiornica', 'meduza',
    'struś', 'paw', 'papuga', 'kanarek', 'kogut', 'bażant', 'kuropatwa', 'przepiórka', 'dzięcioł', 'kukułka',
    'słowik', 'skowronek', 'mewa', 'pelikan', 'flaming', 'anakonda', 'boa', 'kobra', 'pyton', 'żmija',

    # Jedzenie i napoje
    'chleb', 'masło', 'ser', 'mleko', 'woda', 'sok', 'jabłko', 'gruszka', 'śliwka', 'truskawka',
    'malina', 'jagoda', 'ziemniak', 'pomidor', 'ogórek', 'cebula', 'czosnek', 'marchew', 'pietruszka', 'seler',
    'por', 'kapusta', 'sałata', 'rzodkiewka', 'mięso', 'kurczak', 'wieprzowina', 'wołowina', 'ryba', 'sól',
    'pieprz', 'cukier', 'mąka', 'ryż', 'makaron', 'kasza', 'jajko', 'śniadanie', 'obiad', 'kolacja',
    'deser', 'zupa', 'ciasto', 'lody', 'czekolada', 'cukierek', 'lizak', 'ciastko', 'tort', 'wino',
    'piwo', 'wódka', 'szampan', 'koniak', 'likier', 'rumianek', 'kefir', 'kakao', 'kompot', 'lemoniada',
    'orzech', 'migdał', 'rodzynki', 'daktyle', 'figi', 'banan', 'pomarańcza', 'mandarynka', 'cytryna', 'grapefruit',
    'kiwi', 'ananas', 'mango', 'arbuz', 'melon', 'brzoskwinia', 'morela', 'wiśnia', 'czereśnia', 'agrest',
    'porzeczka', 'borówka', 'żurawina', 'papryka', 'dynia', 'cukinia', 'bakłażan', 'brokuł', 'kalafior', 'szpinak',
    'fasola', 'groch', 'soczewica', 'bób', 'koper', 'bazylia', 'oregano', 'tymianek', 'rozmaryn', 'cynamon',

    # Dom, budynki i przedmioty codziennego użytku
    'okno', 'drzwi', 'ściana', 'podłoga', 'sufit', 'dach', 'pokój', 'kuchnia', 'łazienka', 'sypialnia',
    'salon', 'korytarz', 'piwnica', 'strych', 'garaż', 'schody', 'balkon', 'taras', 'ogród', 'płot',
    'brama', 'klucz', 'zamek', 'klamka', 'dzwonek', 'stół', 'krzesło', 'fotel', 'kanapa', 'łóżko',
    'szafa', 'komoda', 'półka', 'biurko', 'dywan', 'obraz', 'lustro', 'zegar', 'telewizor', 'radio',
    'lodówka', 'pralka', 'zmywarka', 'kuchenka', 'mikrofalówka', 'piekarnik', 'odkurzacz', 'żelazko', 'deska', 'garnek',
    'patelnia', 'talerz', 'kubek', 'szklanka', 'sztućce', 'nóż', 'widelec', 'łyżka', 'miska', 'dzbanek',
    'długopis', 'ołówek', 'gumka', 'linijka', 'zeszyt', 'książka', 'kartka', 'papier', 'koperta', 'znaczek',
    'gazeta', 'czasopismo', 'notes', 'kalendarz', 'teczka', 'nożyczki', 'klej', 'taśma', 'spinacz', 'pinezka',
    'worek', 'torba', 'pudełko', 'karton', 'butelka', 'puszka', 'słoik', 'tuba', 'beczka', 'wiadro',
    'szczotka', 'miotła', 'mop', 'gąbka', 'ścierka', 'ręcznik', 'mydło', 'szampon', 'pasta', 'krem',

    # Ciało i ubrania
    'głowa', 'włosy', 'twarz', 'oko', 'ucho', 'nos', 'usta', 'ząb', 'język', 'warga',
    'szyja', 'ramię', 'ręka', 'palec', 'kciuk', 'paznokieć', 'klatka', 'pierś', 'brzuch', 'plecy',
    'kręgosłup', 'noga', 'kolano', 'stopa', 'pięta', 'skóra', 'kość', 'krew', 'mięsień', 'serce',
    'płuco', 'żołądek', 'wątroba', 'nerka', 'mózg', 'jelito', 'żyła', 'tętnica', 'nerw', 'staw',
    'czaszka', 'szczęka', 'broda', 'policzek', 'czoło', 'brew', 'rzęsa', 'powieka', 'łokieć', 'nadgarstek',
    'biodro', 'udo', 'łydka', 'kostka', 'gardło', 'ubranie', 'koszula', 'bluzka', 'sweter', 'bluza',
    'spodnie', 'dżinsy', 'spódnica', 'sukienka', 'kurtka', 'płaszcz', 'czapka', 'szalik', 'rękawiczka', 'but',
    'skarpetka', 'rajstopy', 'bielizna', 'majtki', 'biustonosz', 'krawat', 'pasek', 'torebka', 'plecak', 'portfel',
    'okulary', 'zegarek', 'pierścionek', 'naszyjnik', 'kolczyk', 'bransoletka', 'garnitur', 'kalesony', 'szlafrok', 'piżama',
    'kapelusz', 'kask', 'sandały', 'kozak', 'kalosz', 'kamizelka', 'kaptur', 'guzik', 'suwak', 'kieszeń',

    # Natura, czas i zjawiska geograficzne
    'słońce', 'księżyc', 'gwiazda', 'niebo', 'chmura', 'deszcz', 'śnieg', 'wiatr', 'burza', 'mgła',
    'lód', 'mróz', 'ciepło', 'zimno', 'ogień', 'ziemia', 'powietrze', 'piasek', 'kamień', 'skała',
    'trawa', 'liść', 'gałąź', 'korzeń', 'pień', 'krzew', 'mech', 'grzyb', 'morze', 'ocean',
    'jezioro', 'staw', 'strumień', 'dolina', 'pagórek', 'szczyt', 'wyspa', 'plaża', 'wybrzeże', 'pustynia',
    'dżungla', 'bór', 'piorun', 'błyskawica', 'grzmot', 'tęcza', 'huragan', 'tornado', 'powódź', 'trzęsienie',
    'sekunda', 'minuta', 'godzina', 'dzień', 'noc', 'rano', 'wieczór', 'południe', 'północ', 'tydzień',
    'miesiąc', 'rok', 'wiek', 'poniedziałek', 'wtorek', 'środa', 'czwartek', 'piątek', 'sobota', 'niedziela',
    'styczeń', 'luty', 'marzec', 'kwiecień', 'maj', 'czerwiec', 'lipiec', 'sierpień', 'wrzesień', 'październik',
    'listopad', 'grudzień', 'wiosna', 'lato', 'jesień', 'zima', 'wczoraj', 'dzisiaj', 'jutro', 'przedwczoraj',
    'pojutrze', 'teraz', 'zaraz', 'potem', 'nigdy', 'zawsze', 'często', 'rzadko', 'czasem', 'wkrótce',

    # Miasto, transport, praca i rodzina
    'auto', 'pociąg', 'tramwaj', 'autobus', 'trolejbus', 'metro', 'prom', 'łódź', 'żaglówka', 'helikopter',
    'rakieta', 'motocykl', 'skuter', 'hulajnoga', 'rolki', 'wrotki', 'deskorolka', 'bilet', 'stacja', 'przystanek',
    'dworzec', 'lotnisko', 'port', 'ulica', 'droga', 'autostrada', 'chodnik', 'ścieżka', 'skrzyżowanie', 'rondo',
    'most', 'tunel', 'wiadukt', 'wypadek', 'korek', 'sklep', 'apteka', 'piekarnia', 'rzeźnik', 'warzywniak',
    'market', 'galeria', 'teatr', 'muzeum', 'biblioteka', 'szkoła', 'przedszkole', 'uniwersytet', 'szpital', 'przychodnia',
    'bank', 'poczta', 'policja', 'straż', 'kościół', 'cmentarz', 'park', 'plac', 'pomnik', 'fontanna',
    'restauracja', 'kawiarnia', 'pub', 'hotel', 'basen', 'stadion', 'boisko', 'siłownia', 'klub', 'praca',
    'szef', 'pracownik', 'biuro', 'fabryka', 'firma', 'pensja', 'lekarz', 'pielęgniarka', 'nauczyciel', 'uczeń',
    'student', 'inżynier', 'architekt', 'prawnik', 'sędzia', 'policjant', 'strażak', 'żołnierz', 'rolnik', 'górnik',
    'rodzina', 'matka', 'ojciec', 'mama', 'tata', 'syn', 'córka', 'brat', 'siostra', 'dziadek',

    # Emocje, pojęcia abstrakcyjne, narzędzia i sztuka
    'babcia', 'wnuk', 'wnuczka', 'wujek', 'ciocia', 'kuzyn', 'kuzynka', 'mąż', 'żona', 'teść',
    'miłość', 'nienawiść', 'radość', 'smutek', 'strach', 'złość', 'gniew', 'zaskoczenie', 'zdziwienie', 'nadzieja',
    'wiara', 'szczęście', 'pech', 'ból', 'zdrowie', 'choroba', 'życie', 'śmierć', 'pokój', 'wojna',
    'wolność', 'niewola', 'prawda', 'kłamstwo', 'dobro', 'zło', 'piękno', 'brzydota', 'mądrość', 'głupota',
    'siła', 'słabość', 'odwaga', 'tchórzostwo', 'duma', 'wstyd', 'wina', 'kara', 'nagroda', 'cel',
    'sens', 'marzenie', 'pomysł', 'myśl', 'pamięć', 'uwaga', 'rozum', 'dusza', 'wola', 'charakter',
    'osobowość', 'los', 'przeznaczenie', 'przypadek', 'sukces', 'porażka', 'problem', 'rozwiązanie', 'pytanie', 'odpowiedź',
    'przyczyna', 'skutek', 'początek', 'koniec', 'środek', 'część', 'całość', 'różnica', 'podobieństwo', 'waga',
    'młotek', 'śrubokręt', 'wiertarka', 'piła', 'gwóźdź', 'śruba', 'klucz', 'obcęgi', 'kombinerki', 'gitara',
    'pianino', 'skrzypce', 'flet', 'bęben', 'trąbka', 'saksofon', 'wiolonczela', 'perkusja', 'rytm', 'malarz',

    # Czasowniki (część 1)
    'być', 'mieć', 'móc', 'chcieć', 'musieć', 'wiedzieć', 'mówić', 'robić', 'widzieć', 'iść',
    'dać', 'wziąć', 'spać', 'jeść', 'pić', 'stać', 'siedzieć', 'leżeć', 'biec', 'jechać',
    'latać', 'pływać', 'skakać', 'padać', 'rzucać', 'łapać', 'trzymać', 'nosić', 'ciągnąć', 'pchać',
    'otwierać', 'zamykać', 'zaczynać', 'kończyć', 'szukać', 'znajdować', 'gubić', 'chować', 'pokazywać', 'patrzeć',
    'słuchać', 'słyszeć', 'czuć', 'pachnieć', 'smakować', 'dotykać', 'myśleć', 'pamiętać', 'zapominać', 'rozumieć',
    'uczyć', 'studiować', 'czytać', 'pisać', 'liczyć', 'rysować', 'malować', 'śpiewać', 'tańczyć', 'grać',
    'pracować', 'odpoczywać', 'bawić', 'śmiać', 'płakać', 'cieszyć', 'martwić', 'złościć', 'bać', 'kochać',
    'lubić', 'nienawidzić', 'szanować', 'pomagać', 'przeszkadzać', 'pytać', 'odpowiadać', 'prosić', 'dziękować', 'przepraszać',
    'witać', 'żegnać', 'zapraszać', 'spotykać', 'czekać', 'spieszyć', 'spóźniać', 'zdążyć', 'trwać', 'zmieniać',
    'rosnąć', 'maleć', 'budować', 'niszczyć', 'tworzyć', 'kupować', 'sprzedawać', 'płacić', 'kosztować', 'kraść',

    # Czasowniki (część 2) i Przymiotniki (część 1)
    'oszukiwać', 'walczyć', 'bronić', 'atakować', 'uciekać', 'gonić', 'wygrywać', 'przegrywać', 'rodzić', 'umierać',
    'żyć', 'mieszkać', 'pochodzić', 'nazywać', 'wyglądać', 'znaczyć', 'wydawać', 'zgadzać', 'proponować', 'decydować',
    'dobry', 'zły', 'wielki', 'mały', 'nowy', 'stary', 'młody', 'długi', 'krótki', 'wysoki',
    'niski', 'szeroki', 'wąski', 'gruby', 'chudy', 'ciężki', 'lekki', 'gorący', 'ciepły', 'zimny',
    'chłodny', 'mokry', 'suchy', 'twardy', 'miękki', 'ostry', 'tępy', 'gładki', 'szorstki', 'jasny',
    'ciemny', 'czysty', 'brudny', 'ładny', 'piękny', 'brzydki', 'mądry', 'głupi', 'bogaty', 'biedny',
    'zdrowy', 'chory', 'silny', 'słaby', 'szybki', 'wolny', 'głośny', 'cichy', 'tani', 'drogi',
    'łatwy', 'trudny', 'prosty', 'krzywy', 'pełny', 'pusty', 'wesoły', 'smutny', 'grzeczny', 'niegrzeczny',
    'miły', 'niemiły', 'ciekawy', 'nudny', 'ważny', 'nieważny', 'prawdziwy', 'fałszywy', 'zajęty', 'gotowy',
    'zmęczony', 'głodny', 'spragniony', 'pijany', 'trzeźwy', 'śpiący', 'odważny', 'tchórzliwy', 'dumny', 'skromny',

    # Przymiotniki (część 2), Spójniki, Przyimki, Zaimki i Kolory
    'uczciwy', 'kłamliwy', 'leniwy', 'pracowity', 'spokojny', 'nerwowy', 'ostrożny', 'niebezpieczny', 'bezpieczny', 'dziwny',
    'normalny', 'śmieszny', 'poważny', 'łagodny', 'gorzki', 'słodki', 'kwaśny', 'słony', 'pyszny', 'ohydny',
    'świeży', 'zepsuty', 'wczesny', 'późny', 'pierwszy', 'ostatni', 'kolejny', 'następny', 'poprzedni', 'lewy',
    'prawy', 'górny', 'dolny', 'środkowy', 'główny', 'poboczny', 'biały', 'czarny', 'czerwony', 'niebieski',
    'zielony', 'żółty', 'brązowy', 'pomarańczowy', 'fioletowy', 'różowy', 'szary', 'złoty', 'srebrny', 'bardzo',
    'mało', 'dużo', 'trochę', 'wcale', 'też', 'także', 'oraz', 'lub', 'albo', 'czy',
    'jeśli', 'jeżeli', 'ponieważ', 'dlatego', 'więc', 'zatem', 'jednak', 'ale', 'lecz', 'chociaż',
    'mimo', 'tylko', 'nawet', 'już', 'jeszcze', 'znowu', 'przecież', 'chyba', 'tutaj', 'tam',
    'stąd', 'gdzie', 'kiedy', 'dlaczego', 'kto', 'który', 'czyj', 'mój', 'twój', 'jego',
    'jej', 'nasz', 'wasz', 'ich', 'siebie', 'sobie', 'mną', 'tobą', 'nim', 'nią'

    'alibaba', 'mysz', 'kotek', 'samochod', 'komputer', 'telefon', 'dom', 'drzewo', 'kwiat', 'lampa', 'banany', 'siema', 
    'cesarz', 'informatyka', 'programowanie', 'kot', 'kawa', 
    'herbata', 'rower', 'samolot', 'statek', 'góra', 'rzeka', 'las', 
    'miasto', 'piesek', 'jajeczko', 'matematyka', 'fizyka', 'chemia', 'biologia', 'historia',
    'geografia', 'filozofia', 'psychologia', 'sztuka', 'muzyka', 'sport', 'kino', 'barcelona', 'politechnika', 'marynarz',
    'kropelka', 'polska', 'niemcy', 'włochy', 'francja', 'hiszpania', 'anglia', 'rosja', 'usa', 'japonia', 'chiny', 'indie'
    'ukraina', 'siedemnaście', 'osiemnaście', 'dziewiętnaście', 'dwadzieścia', 'trzydzieści', 'czterdzieści', 'pięćdziesiąt', 'sześćdziesiąt',
    ]


In [2]:
from src.ctc.features import wav_path_to_logmel
from src.ctc.dataset import textgrid_to_phone_ids
from src.ctc.metrics import greedy_decode, decode_to_phones, compute_per

wav_path = "../AutorskieDane/AutorskiDataset/id2.wav"
tg_path = "../AutorskieDane/AutorskiDataset/id2.TextGrid"
wav_path = "../slowa_testowe/alibaba.wav"
#tf_path = "../slowa_testowe/"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

PER for this utterance: 0.9581
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil a l j b a b a sil t r u


In [59]:
wav_path = "../slowa_testowe/politechnika.wav"


target_ids = textgrid_to_phone_ids(tg_path, map_sp_to_sil=True)


mel = wav_path_to_logmel(wav_path)  # (F, T)
mel = mel.unsqueeze(0).to(device)  # (1, F, T)

with torch.no_grad():
    logits = model(mel)  # (1, T', C+1)

pred_ids = greedy_decode(logits.cpu())[0]  # list[int]


per = compute_per([pred_ids], [target_ids])
print(f"PER for this utterance: {per:.4f}")

print("Target:     ", decode_to_phones(target_ids))
print("Prediction: ", decode_to_phones(pred_ids))

decoded = decode_to_phones(pred_ids)
decoded_text = phonemes_to_text(decoded)
w = [str(ph) for ph in decoded_text.split()]
wtext = phonemes_to_text(w, after_silence=False)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    #print(f"Testing word: {wr}")
    dist = levenshtein_distance(wr, w)
    #print(dist)
    if dist < mindist:
        mindist = dist
        output = wr
print("_______________________________________________________________")
print(f"Prediction without correction: {wtext}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")

PER for this utterance: 0.9488
Target:      sil s t u d e n tsj i c e r u n k u sil d o v a d u j oc5 sj e sil j a k sil p o s t e m p o v a tsj sil z d u Z i2 m i i l o sj tsj a m i d a n i2 h sil p o h o dz o n c i2 m i sil z r u Z n i2 h zj r u d e w sil u tS oc5 sj e r u v n~ e S j e i n t e r p r e t o v a tsj sil f S i2 s t k o t o sil v o p a r tsj u o sil p r o g r a m i2 sil i sil a l g o r i2 t m i2 k o m p u t e r o v e sil a sil t a k S e sil v e dz e sil s m a t e m a t i2 c i sil s t a t i2 s t i2 c i sil i e k o n o m j i sil
Prediction:  sil p o l j t e sj n i k a sil
_______________________________________________________________
Prediction without correction: poljteśnika
Best match:  ---- politechnika ----- with distance 5
